In [2]:
from helpers import *
import pandas as pd
csv_df = pd.read_csv('customers_sales.csv')

display(csv_df)


,customer_name,customer_branch,file_name,order_quantity
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800
...,...,...,...,...
74,AutoParts AG,Munich Branch,Geo (78).DXF,8150
75,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750
76,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050
77,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100


In [3]:
customer_branch_id = {}


#print(customer_branch_id)
#csv_df['customer_id'] = csv_df['customer_name'].map(customer_id)
#csv_df['branch_id'] = csv_df['customer_branch'].map(branch_id)

display(csv_df)

,customer_name,customer_branch,file_name,order_quantity
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800
...,...,...,...,...
74,AutoParts AG,Munich Branch,Geo (78).DXF,8150
75,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750
76,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050
77,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100


In [44]:
csv_df = pd.read_csv('customers_sales.csv')

folder = "10991360/"

cutting_speed_mm_min = 1000 # mm/min
offset_time_smartgate_s = 2 #s
offset_time_sortmaster_s = 5 #s
offset_time_hole_s = 3 #s

smartgate_min = (30, 30) # min
smartgate_max = (160, 160) # max

sortmaster_min = (60, 90) # min
sortmaster_max = (1500, 2000) # max

def calculate_cutting_time(row):
    cutting_length = row['cutting_length_mm']
    number_of_holes = row['number_of_holes']
    
    cutting_time_min = cutting_length / cutting_speed_mm_min
    offset_time_total_s = offset_time_smartgate_s + offset_time_sortmaster_s + (number_of_holes * offset_time_hole_s)
    
    total_time_s = cutting_time_min / 60 + offset_time_total_s
    return total_time_s

def get_fit(object, min_opening, max_opening):
    ob_1, ob_2 = sorted(object)
    min_1, min_2 = sorted(min_opening)
    max_1, max_2 = sorted(max_opening)

    return min_1 <= ob_1 <= max_1 and min_2 <= ob_2 <= max_2

def get_extraction_method(file_name):
    bounding_box = tuple(sorted(get_dxf_bound(folder + file_name)))

    if get_fit(bounding_box, sortmaster_min, sortmaster_max) and get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate + Master"
    
    elif get_fit(bounding_box, smartgate_min, smartgate_max):
        return "Gate"
    
    elif get_fit(bounding_box, sortmaster_min, sortmaster_max):
        return "Master"
    
    else:
        return "None"


csv_df['cutting_length_mm'] = csv_df["file_name"].map(lambda x: get_cutting_length(folder + x))
csv_df['bounding_box_width_mm'] = csv_df["file_name"].map(lambda x: get_dxf_bound(folder + x)[0])
csv_df['bounding_box_height_mm'] = csv_df["file_name"].map(lambda x: get_dxf_bound(folder + x)[1])
csv_df['bounding_box_area_mm^2'] = csv_df["file_name"].map(lambda x: get_dxf_bound(folder + x)[0] * get_dxf_bound(folder + x)[1])
csv_df['extraction_method'] = csv_df["file_name"].map(lambda x: get_extraction_method(x))
csv_df['number_of_holes'] = csv_df["file_name"].map(lambda x: get_number_of_holes(folder + x))
csv_df['cutting_time_s'] = csv_df.apply(calculate_cutting_time, axis=1)


In [41]:
pd.set_option('display.max_rows', 10) 
display(csv_df)

,customer_name,customer_branch,file_name,order_quantity,cutting_length_mm,bounding_box_width_mm,bounding_box_height_mm,bounding_box_area_mm^2,extraction_method,number_of_holes,cutting_time_s
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,3438.024977,1600.219048,305.139765,4.882905e+05,Master,0,7.057300
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,3576.198184,853.830303,333.401400,2.846682e+05,Master,0,7.059603
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,1965.579365,164.000000,145.120247,2.379972e+04,Master,0,7.032760
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,1302.774283,193.800891,138.442589,2.683030e+04,Master,0,7.021713
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,2815.159795,937.150697,74.006665,6.935540e+04,Master,9,34.046919
...,...,...,...,...,...,...,...,...,...,...,...
74,AutoParts AG,Munich Branch,Geo (78).DXF,8150,5751.264292,386.361049,290.000000,1.120447e+05,Master,51,160.095854
75,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750,147.032435,36.553638,36.553638,1.336168e+03,Gate,5,22.002451
76,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050,6111.748725,801.705747,344.180571,2.759315e+05,Master,6,25.101862
77,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100,16821.059941,1968.901094,832.434782,1.638982e+06,Master,11,40.280351


In [38]:
#csv_df = csv_df.sort_values(by="order_quantity", ascending=False)
#csv_df = csv_df.sort_values(by="cutting_length_mm", ascending=False)
#csv_df = csv_df.sort_values(by="order_quantity", ascending=False)
#csv_df = csv_df.sort_values(by="order_quantity", ascending=False)
display(csv_df)


,customer_name,customer_branch,file_name,order_quantity,cutting_length_mm,bounding_box_width_mm,bounding_box_height_mm,bounding_box_area_mm^2,extraction_method,number_of_holes,cutting_time_s
0,MetalForm SA,Frankfurt Branch,Geo (1).DXF,1600,3438.024977,1600.219048,305.139765,4.882905e+05,Master,0,7.057300
1,AutoParts AG,Munich Branch,Geo (10).DXF,9650,3576.198184,853.830303,333.401400,2.846682e+05,Master,0,7.059603
2,PrecisionCut GmbH,Stuttgart Branch,Geo (11).DXF,7800,1965.579365,164.000000,145.120247,2.379972e+04,Master,0,7.032760
3,PrecisionCut GmbH,Stuttgart Branch,Geo (15).DXF,4000,1302.774283,193.800891,138.442589,2.683030e+04,Master,0,7.021713
4,AutoParts AG,Berlin Branch,Geo (12).DXF,8800,2815.159795,937.150697,74.006665,6.935540e+04,Master,9,34.046919
...,...,...,...,...,...,...,...,...,...,...,...
74,AutoParts AG,Munich Branch,Geo (78).DXF,8150,5751.264292,386.361049,290.000000,1.120447e+05,Master,51,160.095854
75,PrecisionCut GmbH,Stuttgart Branch,Geo (79).DXF,3750,147.032435,36.553638,36.553638,1.336168e+03,Gate,5,22.002451
76,TechFab Industries,Hamburg Branch,Geo (8).DXF,12050,6111.748725,801.705747,344.180571,2.759315e+05,Master,6,25.101862
77,MetalForm SA,Frankfurt Branch,Geo (80).DXF,2100,16821.059941,1968.901094,832.434782,1.638982e+06,Master,11,40.280351
